In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from scipy.sparse import coo_matrix
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [2]:
columns = ['author', 'subreddit', 'created_utc']
chunks = []
#expect 30s/Gb
for chunk in pd.read_json(
    'RC_2014-12',
    #'RC_2011-12',
    lines=True,
    chunksize=100000
):
    chunks.append(chunk[columns])

initdata = pd.concat(chunks, ignore_index=True)
#initdata = uinitdata [['author', 'subreddit', 'created_utc']]

In [3]:
#cleaning initial data
rows = initdata[initdata["author"] == "[deleted]"].index

initdata.drop(rows, inplace=True)
print("rows")
print(len(initdata))

print("subreddits")
print(initdata['subreddit'].unique())
print(len(initdata['subreddit'].unique()))

print("individual commenters")
print(initdata['author'].unique())
print(len(initdata['author'].unique()))

print(initdata.head())
#40k users over 1k subreddits

rows
44661700
subreddits
['Guildwars2' 'pokemontrades' 'pics' ... 'InnocentVictimReports'
 'libertariancooking' 'RepaintedSinner']
42174
individual commenters
['eror11' 'HaploPaithan' 'brad1775' ... 'LeoCarryo' 'justx123'
 'andyhacker']
2376028
         author           subreddit  created_utc
0        eror11          Guildwars2   1417392000
1  HaploPaithan       pokemontrades   1417392000
2      brad1775                pics   1417392000
3         JC713               apple   1417392000
4     BigBank41  MaddenUltimateTeam   1417392000


In [4]:
gunscars = ['cars','motorcycles','Autos','cityPorn','TopGear','Justrolledintotheshop','MiliataryPorn','carporn','knives','aviation']
sport = ['nfl','sports','soccer','nba','hiphopheads','hockey','baseball','CFB','MMA','fantasyfootball']
toxic = ['Showerthoughts','mildlyinfuriating','pcmasterrrace','TubmlrInAction','Unexpected','thatHappened','CrazyIdeas','FanTheories','TrollXChromosomes','InternetIsBeautiful']
tech = ['talesfromtechsupport','web_design','learnprogramming','techsupportgore','sysadmin','investing','google','SOPA','Ubuntu','Entrepreneur']
drama = ['AskHistorian','ShitRedditSays','Foodforthought','conspiratard','TheoryOfReddit','DepthHub','Enhancement','circlebroke','Feminism','fifthworldproblems']
webcult = ['gifs','LifeProTips','mildlyinteresting','4chan','cringepics','woahdude','JusticePorn','ImGoingToHell','reactiongifs','cringe']
overseas = ['unitedkingdom','dayz','australia','civ','europe','KerbalSpaceProgram','Eve','britishproblems','Planetside','polandball']
pol = ['Conservative','PoliticalDiscussion','ronpaul','TrueAtheism','DebateReligion','islam','progressive','PoliticalHumor','Republican','POLITIC']
geek = ['offbeat','DoesAnybodyElse','programming','comics','self','geek','entertainment','scifi','apple','business']
porn = ['RealGirls','NSFW_GIF','Boobies','celebs','ass','PrettyGirls','japan','girlsinyogapants','nsfw_gifs','milf']
drugbt = ['Bitcoin','LucidDreaming','Psychonaut','treecomics','Paranormal','UFOs','dogecoin','Glitch_in_the_matrix','classicalmusic','punk']
music = ['lisentothis','WeAreTheMusicMakers','Guitar','dubstep','Metal','electronicmusic','vinyl','ifyoulikeblank','classicalmusic','punk']
gaming = ['Games','Skyrim','buildapc','gameofthrones','pokemon','breakingbad','leagueoflegends','starcraft','techsupport','tf2']
diy = ['food','Frugal','TwoXChromosomes','TrueReddit','malefashionadvice','DIY','firstworldproblems','YouShouldKnow','loseit','Cooking']
defa = ['AskReddit','pics','funny','WTF','gaming','IAmA','todayilearned','videos','politics','worldnews']
supp = ['relationships','relationship_advice','amiugly','AskMen','seduction','AskWomen','depression','confession','SuicideWatch','OkCupid']
conspiracy = ['conspiracy']


subredditlist= gunscars + sport+ toxic+tech+drama+webcult+overseas+pol+geek+porn+drugbt+music+gaming+diy+defa+supp+conspiracy



filterinitdata = initdata#[initdata['subreddit'].isin(subredditlist)]

#easy filtering cuts out 1/3 of the data, still highly significant
print(len(filterinitdata))


44661700


In [5]:
"""
mapping = {}

for name, group in [
    ('gunscars', gunscars),
    ('sport', sport),
    ('toxic', toxic),
    ('tech', tech),
    ('drama', drama),
    ('webcult', webcult),
    ('overseas', overseas),
    ('pol', pol),
    ('geek', geek),
    ('porn', porn),
    ('drugbt', drugbt),
    ('music', music),
    ('gaming', gaming),
    ('diy', diy),
    ('defa', defa),
    ('supp', supp),
    ('conspiracy', conspiracy),
]:
    for sub in group:
        mapping[sub] = name

# Apply vectorized mapping

taggeddata = filterinitdata
taggeddata["community"] = taggeddata["subreddit"].map(mapping)
print(taggeddata.head())
"""

'\nmapping = {}\n\nfor name, group in [\n    (\'gunscars\', gunscars),\n    (\'sport\', sport),\n    (\'toxic\', toxic),\n    (\'tech\', tech),\n    (\'drama\', drama),\n    (\'webcult\', webcult),\n    (\'overseas\', overseas),\n    (\'pol\', pol),\n    (\'geek\', geek),\n    (\'porn\', porn),\n    (\'drugbt\', drugbt),\n    (\'music\', music),\n    (\'gaming\', gaming),\n    (\'diy\', diy),\n    (\'defa\', defa),\n    (\'supp\', supp),\n    (\'conspiracy\', conspiracy),\n]:\n    for sub in group:\n        mapping[sub] = name\n\n# Apply vectorized mapping\n\ntaggeddata = filterinitdata\ntaggeddata["community"] = taggeddata["subreddit"].map(mapping)\nprint(taggeddata.head())\n'

In [6]:
"""taggeddata['community'].value_counts()"""


"taggeddata['community'].value_counts()"

In [7]:
"""
# Count per author per community
counts = taggeddata.groupby(['author', 'subreddit']).size().unstack(fill_value=0)

# Add total comments per author
counts['total_comments'] = counts.sum(axis=1)

# Optional: reset index if you want author as a column
counts = counts.reset_index()

print(counts.head())
print("total authors "+ str(len(counts)))

top_authors = counts[counts['total_comments'] >= 1]

print("top authors " + str(len(top_authors)))
"""

'\n# Count per author per community\ncounts = taggeddata.groupby([\'author\', \'subreddit\']).size().unstack(fill_value=0)\n\n# Add total comments per author\ncounts[\'total_comments\'] = counts.sum(axis=1)\n\n# Optional: reset index if you want author as a column\ncounts = counts.reset_index()\n\nprint(counts.head())\nprint("total authors "+ str(len(counts)))\n\ntop_authors = counts[counts[\'total_comments\'] >= 1]\n\nprint("top authors " + str(len(top_authors)))\n'

In [8]:
"""
# Remove non-community columns
community_matrix = counts.drop(columns=['author', 'total_comments'], errors='ignore')

# Co-occurrence matrix
co_matrix = community_matrix.T.dot(community_matrix)

binary = (community_matrix > 0).astype(int)
co_matrix_binary = binary.T.dot(binary)
co_matrix = co_matrix_binary

print(co_matrix.head())
co_matrix = np.log10(co_matrix + 1)
co_matrix = co_matrix / np.sqrt(np.outer(np.diag(co_matrix), np.diag(co_matrix)))
print(co_matrix.head())


print(co_matrix_binary.head())
"""

"\n# Remove non-community columns\ncommunity_matrix = counts.drop(columns=['author', 'total_comments'], errors='ignore')\n\n# Co-occurrence matrix\nco_matrix = community_matrix.T.dot(community_matrix)\n\nbinary = (community_matrix > 0).astype(int)\nco_matrix_binary = binary.T.dot(binary)\nco_matrix = co_matrix_binary\n\nprint(co_matrix.head())\nco_matrix = np.log10(co_matrix + 1)\nco_matrix = co_matrix / np.sqrt(np.outer(np.diag(co_matrix), np.diag(co_matrix)))\nprint(co_matrix.head())\n\n\nprint(co_matrix_binary.head())\n"

In [9]:

"""

plt.figure(figsize=(12, 10))
plt.imshow(co_matrix, aspect='auto')

plt.colorbar(label='User activity overlap')

plt.xticks(range(len(co_matrix.columns)), co_matrix.columns, rotation=90)
plt.yticks(range(len(co_matrix.index)), co_matrix.index)

plt.title('Community Overlap Heatmap')

plt.tight_layout()
plt.show()
"""

"\n\nplt.figure(figsize=(12, 10))\nplt.imshow(co_matrix, aspect='auto')\n\nplt.colorbar(label='User activity overlap')\n\nplt.xticks(range(len(co_matrix.columns)), co_matrix.columns, rotation=90)\nplt.yticks(range(len(co_matrix.index)), co_matrix.index)\n\nplt.title('Community Overlap Heatmap')\n\nplt.tight_layout()\nplt.show()\n"

In [10]:
"""
# Convert users/subreddits to categorical integer IDs
author_codes = taggeddata['author'].astype('category')
subreddit_codes = taggeddata['subreddit'].astype('category')

rows = author_codes.cat.codes
cols = subreddit_codes.cat.codes

# One entry per interaction
data = np.ones(len(taggeddata), dtype=np.uint8)

# Sparse user × subreddit matrix
matrix = coo_matrix(
    (data, (rows, cols)),
    shape=(
        len(author_codes.cat.categories),
        len(subreddit_codes.cat.categories)
    )
).tocsr()

# Log transform ONLY nonzero entries
matrix.data = np.log1p(matrix.data)

# Save subreddit names
subreddit_names = subreddit_codes.cat.categories

# TF-IDF
tfidf = TfidfTransformer()
matrix_tfidf = tfidf.fit_transform(matrix)
"""

"\n# Convert users/subreddits to categorical integer IDs\nauthor_codes = taggeddata['author'].astype('category')\nsubreddit_codes = taggeddata['subreddit'].astype('category')\n\nrows = author_codes.cat.codes\ncols = subreddit_codes.cat.codes\n\n# One entry per interaction\ndata = np.ones(len(taggeddata), dtype=np.uint8)\n\n# Sparse user × subreddit matrix\nmatrix = coo_matrix(\n    (data, (rows, cols)),\n    shape=(\n        len(author_codes.cat.categories),\n        len(subreddit_codes.cat.categories)\n    )\n).tocsr()\n\n# Log transform ONLY nonzero entries\nmatrix.data = np.log1p(matrix.data)\n\n# Save subreddit names\nsubreddit_names = subreddit_codes.cat.categories\n\n# TF-IDF\ntfidf = TfidfTransformer()\nmatrix_tfidf = tfidf.fit_transform(matrix)\n"

In [11]:
"""
# SVD
ncomps = 250
svd = TruncatedSVD(
    n_components=ncomps,
    random_state=42
)

nc_svd = svd.fit_transform(matrix_tfidf)


# COMPONENTS DATAFRAME
components = pd.DataFrame(
    svd.components_,
    columns=subreddit_names
)


# PRINT COMPONENTS
for i in range(ncomps):
    comp = components.iloc[i]
    print(f"\n======================")
    print(f"Component {i}")
    print(f"Explained variance: {svd.explained_variance_ratio_[i]:.4f}")
    print("\nTop positive:")
    print(
        comp.sort_values(ascending=False).head(10)
    )
    print("\nTop negative:")
    print(
        comp.sort_values().head(10)
    )


# SAVE TO FILE
with open("file.txt", "w", encoding="utf-8") as f:
    for i in range(ncomps):
        comp = components.iloc[i]
        f.write("\n====================================\n")
        f.write(f"Component {i}\n")
        f.write(
            f"Explained variance: "
            f"{svd.explained_variance_ratio_[i]:.6f}\n"
        )
        f.write("\nTop positive:\n")
        f.write(
            comp.sort_values(ascending=False)
            .head(100)
            .to_string()
        )
        f.write("\n\nTop negative:\n")
        f.write(
            comp.sort_values()
            .head(100)
            .to_string()
        )
        f.write("\n\n")
"""

'\n# SVD\nncomps = 250\nsvd = TruncatedSVD(\n    n_components=ncomps,\n    random_state=42\n)\n\nnc_svd = svd.fit_transform(matrix_tfidf)\n\n\n# COMPONENTS DATAFRAME\ncomponents = pd.DataFrame(\n    svd.components_,\n    columns=subreddit_names\n)\n\n\n# PRINT COMPONENTS\nfor i in range(ncomps):\n    comp = components.iloc[i]\n    print(f"\n======================")\n    print(f"Component {i}")\n    print(f"Explained variance: {svd.explained_variance_ratio_[i]:.4f}")\n    print("\nTop positive:")\n    print(\n        comp.sort_values(ascending=False).head(10)\n    )\n    print("\nTop negative:")\n    print(\n        comp.sort_values().head(10)\n    )\n\n\n# SAVE TO FILE\nwith open("file.txt", "w", encoding="utf-8") as f:\n    for i in range(ncomps):\n        comp = components.iloc[i]\n        f.write("\n====================================\n")\n        f.write(f"Component {i}\n")\n        f.write(\n            f"Explained variance: "\n            f"{svd.explained_variance_ratio_[i]:.6f}

Component 72subreddit
conspiracy       0.954546
worldpolitics    0.075702
MensRights       0.053983
shittyadvice     0.039144
Anarchism        0.032908
itookapicture    0.032612
UFOs             0.027814
listentothis     0.020141
911truth         0.019576
wow              0.019420
ronpaul          0.013698
collapse         0.013022
Guitar           0.012965
Cooking          0.011717
Israel           0.011333
religion         0.011136
Art              0.011073
gadgets          0.010978
aww              0.010944
skeptic          0.010387

conspiracy matrix, unfiltered communities, ufos, 911truth are defnitely conspiratorial, israel, religion, ronpaul, and anarchism are conspi-adjacent depending on mods and time period


In [12]:
#ml sequence, independent from everything above, but initdata

df = initdata.sort_values(
    ['author', 'created_utc']
).copy()

consubs = 'conspiracy', 'UFOs', '911truth', 'conspiracies'

#climateskeptics, Paranormal, skeptics, ronpaul
#all subreddits that have been set aside due to beinc potentially conspiratorial

feature_parts = []
labels = {}

for author, g in df.groupby('author', sort=False):


    g = g.sort_values('created_utc').reset_index(drop=True)

    # Find first conspiracy comment
    con_mask = g['subreddit'].isin(consubs)

    if con_mask.any():

        first_con_idx = np.flatnonzero(con_mask)[0]

        # Need at least 5 comments before first conspiracy
        if first_con_idx < 5:
            continue

        history = g.iloc[:first_con_idx ]
        future = g.iloc[first_con_idx :first_con_idx+5]

        label = 1

    else:

        # Never enters conspiracy subs
        if len(g) < 6:
            continue

        history = g.iloc[:-5]
        future = g.iloc[-5:]

        label = 0

    if len(history) == 0:
        continue
    labels[author] = label
    feature_parts.append(history)

train_df = pd.concat(
    feature_parts,
    ignore_index=True
)

authors = pd.DataFrame.from_dict(
    labels,
    orient='index',
    columns=['label']
)

In [13]:
#slow, expect 20s/Gb
min_comments = 20
#filtering tiny subreddits
valid_subs = (
    train_df['subreddit']
    .value_counts()
)
valid_subs = valid_subs[
    valid_subs >= min_comments
].index
train_df = train_df[
    train_df['subreddit'].isin(valid_subs)
]



author_codes = train_df['author'].astype('category')
subreddit_codes = train_df['subreddit'].astype('category')


subreddit_names = subreddit_codes.cat.categories

rows = author_codes.cat.codes
cols = subreddit_codes.cat.codes



data = np.ones(len(train_df), dtype=np.uint8)
#generate matrix
X = coo_matrix(
    (data, (rows, cols)),
    shape=(
        len(author_codes.cat.categories),
        len(subreddit_codes.cat.categories)
    )
).tocsr()

#cleanup
X.sum_duplicates()


unweightcomments=False
if unweightcomments:
    #make data binary by activity
    X.data[:] = 1
else:
    #reduce activity weight
    X.data = np.log1p(X.data)

tfidf = TfidfTransformer(use_idf=True)

X = tfidf.fit_transform(X)



#label stuff
author_index = pd.Index(author_codes.cat.categories)

y = (
    authors
    .reindex(author_index)['label']
    .fillna(0)
    .astype(np.int8)
)



X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

#efficiency
use_svd = True

if use_svd:

    svd = TruncatedSVD(
        n_components=100,
        random_state=42
    )

    X_train = svd.fit_transform(X_train)
    X_test = svd.transform(X_test)


#run the model
model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [14]:

probs = model.predict_proba(X_test)[:, 1]

threshold = 0.90

pred = (probs >= threshold).astype(np.int8)

print(classification_report(y_test, pred))



              precision    recall  f1-score   support

           0       0.99      0.98      0.99    193617
           1       0.03      0.09      0.05      1165

    accuracy                           0.98    194782
   macro avg       0.51      0.54      0.52    194782
weighted avg       0.99      0.98      0.98    194782



In [15]:
subreddit_weights = (
    model.coef_ @ svd.components_
).flatten()

coef = pd.Series(
    subreddit_weights,
    index=subreddit_names
)

print(coef.sort_values(ascending=False).head(10))
print(coef.sort_values(ascending=False).tail(5))

politics          4.006215
technology        3.240514
worldnews         3.057363
news              2.967166
Bitcoin           2.605546
atheism           2.601713
australia         2.149167
TumblrInAction    2.039784
fatpeoplehate     2.019048
todayilearned     1.993128
dtype: float64
bravefrontier     -2.477489
friendsafari      -2.523939
MakeupAddiction   -2.773986
pokemontrades     -3.179056
csgobetting       -3.965388
dtype: float64


politics             2.939206
news                 2.507942
worldnews            2.254505
technology           2.174900
atheism              1.909383
todayilearned        1.720771
WTF                  1.597520
TumblrInAction       1.329963
videos               1.254879
AdviceAnimals        1.221388
Bitcoin              1.097564
MMA                  0.953125
movies               0.938424
explainlikeimfive    0.884298
creepy               0.767815
trees                0.753478
Showerthoughts       0.685883
nfl                  0.663606
gifs                 0.656721
gaming               0.656480
dtype: float64

In [16]:
scores = model.predict_proba(X_test)[:, 1]

ranking = pd.DataFrame({
    "y": y_test,
    "score": scores
}).sort_values("score", ascending=False)

ranking.head(100)["y"].mean()

np.float64(0.06)